In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

# Chargement du dataset nettoyé
df = pd.read_csv('../data/data_clean.csv', parse_dates=['InvoiceDate'])

print(f"Dataset chargé : {df.shape}")
print(f"\npériode : du {df['InvoiceDate'].min()} au {df['InvoiceDate'].max()}")

Dataset chargé : (397884, 9)

période : du 2010-12-01 08:26:00 au 2011-12-09 12:50:00


In [4]:
# Date de référence = lendemain de la dernière transaction 
reference_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f"Date de référence : {reference_date}")

# Calcul RFM par client
rfm = df.groupby('CustomerID').agg(
    Recence = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequence=('InvoiceNo', 'nunique'),
    Montant = ('TotalPrice', 'sum')
).reset_index()

print(f"\nNombre de clients analysés : {len(rfm)}")
print(f"\nAperçu RFM :")
print(rfm.head(10))
print(f"\nStatistiques :")
print(rfm.describe())

Date de référence : 2011-12-10 12:50:00

Nombre de clients analysés : 4338

Aperçu RFM :
   CustomerID  Recence  Frequence   Montant
0       12346      326          1  77183.60
1       12347        2          7   4310.00
2       12348       75          4   1797.24
3       12349       19          1   1757.55
4       12350      310          1    334.40
5       12352       36          8   2506.04
6       12353      204          1     89.00
7       12354      232          1   1079.40
8       12355      214          1    459.40
9       12356       23          3   2811.43

Statistiques :
         CustomerID      Recence    Frequence        Montant
count   4338.000000  4338.000000  4338.000000    4338.000000
mean   15300.408022    92.536422     4.272015    2054.266460
std     1721.808492   100.014169     7.697998    8989.230441
min    12346.000000     1.000000     1.000000       3.750000
25%    13813.250000    18.000000     1.000000     307.415000
50%    15299.500000    51.000000     2.000000

In [8]:
# Scoring de 1 à 4
rfm['R_score'] = pd.qcut(rfm['Recence'], q=4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequence'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Montant'], q=4, labels=[1,2,3,4])

# Segmentation
def segmenter_client(row):
    r = int(row['R_score'])
    f = int(row['F_score'])
    m = int(row['M_score'])
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champions'
    elif r >= 3 and f >= 2:
        return 'Clients fidèles'
    elif r >= 3 and f == 1:
        return 'Nouveaux clients'
    elif r <= 2 and f >= 3:
        return 'Clients à risque'
    else:
        return 'Clients perdus'

rfm['Segment'] = rfm.apply(segmenter_client, axis=1)

print("Répartition des segments :")
print(rfm['Segment'].value_counts())
print(f"\nPourcentages :")
print(rfm['Segment'].value_counts(normalize=True).mul(100).round(1))

Répartition des segments :
Segment
Clients perdus      1504
Champions           1319
Clients à risque     646
Clients fidèles      610
Nouveaux clients     259
Name: count, dtype: int64

Pourcentages :
Segment
Clients perdus      34.7
Champions           30.4
Clients à risque    14.9
Clients fidèles     14.1
Nouveaux clients     6.0
Name: proportion, dtype: float64


In [9]:
# CA par segment
ca_segment = rfm.groupby('Segment').agg(
    Nombre_clients = ('CustomerID', 'count'),
    CA_total = ('Montant', 'sum'),
    Montant_moyen = ('Montant', 'mean')
).round(2)

ca_segment['Part_CA'] = (ca_segment['CA_total'] / ca_segment['CA_total'].sum()*100).round(1)

print("CA par segment :")
print(ca_segment.sort_values('CA_total', ascending=False))

CA par segment :
                  Nombre_clients    CA_total  Montant_moyen  Part_CA
Segment                                                             
Champions                   1319  6506541.99        4932.94     73.0
Clients à risque             646  1040932.69        1611.35     11.7
Clients perdus              1504   767146.84         510.07      8.6
Clients fidèles              610   504670.83         827.33      5.7
Nouveaux clients             259    92115.55         355.66      1.0


## Résultats de la segmentation RFM

|Segment         | Clients| Part clients | Ca total   | Part CA |
|----------------|--------|--------------|------------|---------|
|Champions       | 1319   |  30.4%       | 6 506 541 £|  73%    |
|Clients à risque| 646    |  14.9%       | 1 040 932 £|  11.7%  |
|Clients perdus  | 1504   |  34.7%       |   767 146 £|  8.6%   |
|Clients fidèles | 610    |  14.1%       |   504 670 £|  5.7%   |
|Nouveaux clients| 259    |  6%          |    92 115 £|  1%     |

**Insight clé : 30% des cleints générent 73% du CA**